# Hackathon ML 2026 : notebook d'équipe

Cible : `intensity` (easy / moderate / hard). Métrique : macro-F1.
Toute la validation passe par `src/validation.py` (5 folds stratifiés, graine 42).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

from src.validation import load_data, evaluate, plot_confusion, make_submission, make_preprocessor, SEED

pd.set_option("display.max_columns", 50)

## 1. Chargement

In [ ]:
train, test, sample = load_data()
ID_COL, TARGET = "workout_id", "intensity"
print(train.shape, test.shape)
train.head()

## 2. Exploration rapide

In [ ]:
train.info()
print("\nValeurs manquantes :")
print(train.isna().mean().sort_values(ascending=False).head(15))

In [ ]:
print(train[TARGET].value_counts(normalize=True).round(3))
train.describe().T

In [ ]:
# Colonnes catégorielles : nombre de modalités et lien avec la cible
cat_cols = [c for c in train.select_dtypes(exclude="number").columns if c != TARGET]
for c in cat_cols:
    print(f"{c}: {train[c].nunique()} modalités")
    if train[c].nunique() <= 15:
        display(pd.crosstab(train[c], train[TARGET], normalize="index").round(2))

In [ ]:
# Variables numériques par classe
num_cols = [c for c in train.select_dtypes(include="number").columns if c != ID_COL]
train.groupby(TARGET)[num_cols].median().T

## 3. Variables

Point de départ : toutes les colonnes sauf l'identifiant et la cible.
Les variables construites par l'équipe s'ajoutent dans `add_features` (appliquée au train **et** au test).

In [ ]:
def add_features(df):
    df = df.copy()
    # TODO équipe : nouvelles variables
    return df

X = add_features(train).drop(columns=[ID_COL, TARGET])
y = train[TARGET]
X_test = add_features(test).drop(columns=[ID_COL])
print(X.shape, X_test.shape)

## 4. Comparaison de 3 familles de modèles (exigée pour le LOG.md)

In [ ]:
models = {
    "logreg": make_pipeline(make_preprocessor(scale=True), LogisticRegression(max_iter=2000)),
    "knn": make_pipeline(make_preprocessor(scale=True), KNeighborsClassifier(n_neighbors=25)),
    "random_forest": make_pipeline(make_preprocessor(scale=False),
                                   RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED)),
    "hist_gb": make_pipeline(make_preprocessor(scale=False), HistGradientBoostingClassifier(random_state=SEED)),
}

results = {}
for name, model in models.items():
    print(f"--- {name}")
    results[name] = evaluate(model, X, y)

pd.DataFrame({k: (v["mean"], v["std"]) for k, v in results.items()}, index=["macro-F1", "std"]).T.sort_values("macro-F1", ascending=False)

## 5. Analyse d'erreur (Gate 2)

In [ ]:
best = max(results, key=lambda k: results[k]["mean"])
print("Meilleur modèle :", best)
plot_confusion(y, results[best]["oof_pred"], title=f"{best} : matrice de confusion hors fold")

In [ ]:
# Exemples mal classés à regarder à la main
errors = train.assign(pred=results[best]["oof_pred"]).query(f"pred != {TARGET}")
print(len(errors), "erreurs")
errors.sample(10, random_state=SEED)

## 6. Soumission

Avant chaque envoi : ajouter une ligne dans `LOG.md`.

In [ ]:
SUB_NUM = 1
sub = make_submission(models[best], X, y, X_test, test[ID_COL], f"submissions/sub_{SUB_NUM:02d}_{best}.csv")
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)